# Explore Calendar DB

## Setup

In [1]:
import os

import pandas as pd

from event_tracking.config import EVENTS_DB_DIR

## Exploration

In [2]:
df = pd.read_parquet(EVENTS_DB_DIR)

In [3]:
df.head()

,event_id,summary,calendar_name,start_time,end_time,all_day,duration_minutes,day_of_week,week_number,day,month,year,hour_of_day,count,event_category,year_only,year_month,year_month_week
0,_8gsk4d216d338ba58coj8b9k68o48b9o74s44b9h70qje...,NUOVO INIZIO 🤞🏻❣️🍀,Giulia,2025-01-07 06:00:00,2025-01-07 08:00:00+01:00,False,60.0,Tuesday,2,7,January,2025,7.0,NaN,None,2025,2025-01,2025-01-02
1,_8p2kcdq174s30b9g6d2jeb9k8orkcba28913ib9l6ssj4...,Mani @ Giuli,Giulia,2025-01-10 17:00:00,2025-01-10 19:00:00+01:00,False,60.0,Friday,2,10,January,2025,18.0,NaN,None,2025,2025-01,2025-01-02
2,_8l142g9i64pj6ba26cs3gb9k6d346ba26crjgba56koja...,Ceretta @ Giuli,Giulia,2025-01-17 15:30:00,2025-01-17 17:30:00+01:00,False,60.0,Friday,3,17,January,2025,16.0,NaN,None,2025,2025-01,2025-01-03
3,_6krk6c9m64o3cb9o6so44b9k8h0j0ba17123gba288pj4...,Pulizia del viso,Giulia,2025-01-17 16:00:00,2025-01-17 18:00:00+01:00,False,60.0,Friday,3,17,January,2025,17.0,NaN,None,2025,2025-01,2025-01-03
4,_8cpj2d1p64r3iba668q42b9k68q4cb9p89232b9j68r48...,Tatuaggio,Giulia,2025-01-18 07:00:00,2025-01-18 10:00:00+01:00,False,120.0,Saturday,3,18,January,2025,8.0,NaN,None,2025,2025-01,2025-01-03


## Statistics

In [4]:
(
    df
    .groupby(['year'])
    .size()
    .to_frame('events')
    .reset_index()
)

,year,events
0,2025,1687


In [5]:
(
    df
    .groupby(['year', 'calendar_name'])
    .size()
    .to_frame('events')
    .reset_index()
)

,year,calendar_name,events
0,2025,Giulia,53
1,2025,Pozz,434
2,2025,Pozz Health & Learn,749
3,2025,Pozz Work,451


In [7]:
(
    df
    .groupby(['calendar_name', 'year_month'])
    .size()
    .to_frame('events')
    .reset_index()
    .pivot_table(index='year_month', columns='calendar_name', values='events')
)

calendar_name,Giulia,Pozz,Pozz Health & Learn,Pozz Work
year_month,,,,
2025-01,9.0,72.0,133.0,87.0
2025-02,5.0,58.0,161.0,70.0
2025-03,7.0,72.0,175.0,72.0
2025-04,9.0,71.0,115.0,61.0
2025-05,14.0,63.0,73.0,76.0
2025-06,8.0,74.0,63.0,64.0
2025-07,1.0,24.0,29.0,21.0


### Multi Scaling Planning

In [9]:
print(df[['start_time', 'year_only', 'year_month', 'year_month_week']].head())

           start_time year_only year_month year_month_week
0 2025-01-07 06:00:00      2025    2025-01      2025-01-02
1 2025-01-10 17:00:00      2025    2025-01      2025-01-02
2 2025-01-17 15:30:00      2025    2025-01      2025-01-03
3 2025-01-17 16:00:00      2025    2025-01      2025-01-03
4 2025-01-18 07:00:00      2025    2025-01      2025-01-03


In [10]:
df.dtypes

event_id                                          object
summary                                           object
calendar_name                                     object
start_time                                datetime64[ns]
end_time            datetime64[us, pytz.FixedOffset(60)]
all_day                                             bool
duration_minutes                                 float64
day_of_week                                       object
week_number                                        int64
day                                                int64
month                                             object
year                                               int64
hour_of_day                                      float64
count                                            float64
event_category                                    object
year_only                                         object
year_month                                        object
year_month_week                

In [12]:
view = "monthly"
dict_view_available = {
    'yearly': 'year_only',
    'monthly': 'year_month',
    'weekly': 'year_month_week'
}

df_pivot = (
    df
    .loc[(df["calendar_name"] == "Pozz Work"), :]
    # .head()
    .groupby([dict_view_available[view], "summary"])
    .size()
    .to_frame("count")
    .sort_values("count", ascending=False)
    .pivot_table(index="summary", columns=dict_view_available[view], values="count")
    .fillna(0)
)

df_pivot.shape

(286, 7)

In [13]:
df_pivot.head()

year_month,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07
summary,,,,,,,
AIM,0.0,0.0,1.0,0.0,0.0,1.0,1.0
AIM CGS explainer,0.0,0.0,0.0,0.0,0.0,1.0,0.0
API affordability,0.0,0.0,3.0,0.0,0.0,0.0,0.0
CPS grecia,0.0,0.0,0.0,0.0,0.0,2.0,0.0
PR,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
import plotly.express as px

df_temp = df_pivot.sample(10)

# Creazione della heatmap
fig = px.imshow(df_temp,
                labels=dict(x="Time", y="Summary", color="Valore"),
                x=df_temp.columns,
                y=df_temp.index,
                color_continuous_scale="Blues") # Puoi scegliere altre scale di colori

fig.update_xaxes(type='category')
fig.update_yaxes(type='category')

# Mostra la figura
fig.show()


In [ ]:
df.head()

### Categorization through LLM

In [ ]:
df_events_work = (
    df
    .loc[(df["calendar_name"] == "Pozz Work"), :]
    # .head()
    .groupby(["summary"])
    .size()
    .to_frame("count")
    .sort_values("count", ascending=False)
    .fillna(0)
    .reset_index()
)

df_events_work.head()

In [ ]:
df_events_work.columns

#### API

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import openai
import pandas as pd

# OpenAI API Key
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

# Definizione delle categorie
categories = ["avm-property-value", "avm-meetings", "avm-genertel-poc",
              "finbox-meetings", "finbox-gara-mcc", "finbox-privati",
              "finbox-deploy-affordability",
              "smart-lending-suite-meetings",
              "side-project-tools-n-pipeline", "dss-best-practices",
              "other"]

In [ ]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


def classify_batch(llm_app, summaries, categories):
    joined = "\n".join([f"{i+1}. {text}" for i, text in enumerate(summaries)])
    prompt = f"""
Classifica ciascun testo nella seguente lista in **una sola** delle categorie seguenti:
{", ".join(categories)}

Ecco i testi da classificare (uno per riga, preceduto dal numero):

{joined}

Rispondi fornendo solo una lista nel formato:

1. categoria
2. categoria
...
"""
    response = llm_app.invoke(prompt)
    lines = response.content.strip().split("\n")
    # Rimuove numerazione e tiene solo le categorie
    return [line.split(". ", 1)[1].strip() for line in lines if ". " in line]

# Parametri
categories = [ "avm-property-value", "avm-meetings", "avm-genertel-poc",
    "finbox-meetings", "finbox-gara-mcc", "finbox-privati",
    "finbox-deploy-affordability", "smart-lending-suite-meetings",
    "side-project-tools-n-pipeline", "dss-best-practices", "other"
]

In [ ]:
%%time

batch_size = 70  # puoi aumentare finché non superi i limiti di token

results = []
for i in range(0, len(df_events_work), batch_size):
    batch = df_events_work.iloc[i:i+batch_size]
    batch_categories = classify_batch(llm, batch["summary"].tolist(), categories)
    results.extend(batch_categories)

# Verifica che lunghezze corrispondano
if len(results) != len(df_events_work):
    raise ValueError(f"Mismatch: {len(results)} results vs {len(df)} rows")

df_events_work["event_category"] = results

In [ ]:
df_events_work.shape

In [ ]:
(
    df_events_work
    .groupby(["event_category"])
    .size()
    .to_frame("events")
    .reset_index()
)

In [ ]:
df.columns

In [ ]:
df_pivot_cat = (
    df
    .loc[(df["calendar_name"] == "Pozz Work"), :]
    # .head()
    .merge(df_events_work, on="summary", how="left")
    #.head()
    .groupby([dict_view_available[view], "event_category"])
    # .size()
    .agg({"duration_minutes": "sum"})
    .sort_values("duration_minutes", ascending=False)
    .pivot_table(index="event_category", columns=dict_view_available[view], values="duration_minutes")
    .fillna(0)
)

In [ ]:
# df_pivot_cat = (
#     df
#     .loc[(df["calendar_name"] == "Pozz Work"), :]
#     # .head()
#     .merge(df_events_work, on="summary", how="left")
#     .groupby([dict_view_available[view], "event_category"])
#     .size()
#     .to_frame("count")
#     .sort_values("count", ascending=False)
#     .pivot_table(index="event_category", columns=dict_view_available[view], values="count")
#     .fillna(0)
# )

In [ ]:
# Creazione della heatmap
fig = px.imshow(df_pivot_cat,
                labels=dict(x="Time", y="Summary", color="Valore"),
                x=df_pivot_cat.columns,
                y=df_pivot_cat.index,
                color_continuous_scale="Blues") # Puoi scegliere altre scale di colori

fig.update_xaxes(type='category')
fig.update_layout(title="Evoluzione attività svolte", title_x=0.5)

# Mostra la figura
fig.show()

#### Local Ollama

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(model="gemma2:2b", temperature=0)

# Semplice test diretto
response = llm.invoke("Quali sono i 3 principali framework di machine learning?")

print("\n--- Risposta diretta ---")
print(response)

In [ ]:
def classify_batch_local(llm_app, summaries, categories):
    joined = "\n".join([f"{i+1}. {text}" for i, text in enumerate(summaries)])
    prompt = f"""
Classifica ciascun testo nella seguente lista in **una sola** delle categorie seguenti:
{", ".join(categories)}

Ecco i testi da classificare (uno per riga, preceduto dal numero):

{joined}

Rispondi fornendo solo una lista nel formato:

1. categoria
2. categoria
...
"""
    response = llm_app.invoke(prompt)
    lines = response.strip().split("\n")
    # Rimuove numerazione e tiene solo le categorie
    return [line.split(". ", 1)[1].strip() for line in lines if ". " in line]

In [ ]:
def classify_batch_local(llm_app, summaries, categories):
    """
    Classifica testi in categorie predefinite usando modelli locali Ollama,
    con miglioramenti per prevenire allucinazioni e garantire risultati completi.

    Args:
        llm_app: Istanza modello Ollama
        summaries: Lista di testi da classificare
        categories: Lista di categorie valide

    Returns:
        Lista di categorie assegnate, una per ogni testo in input
    """
    # Converti le categorie in un formato numerato per maggiore chiarezza
    categories_formatted = "\n".join([f"{i+1}. {cat}" for i, cat in enumerate(categories)])

    # Crea una stringa con i testi da classificare
    texts_to_classify = "\n".join([f"Testo {i+1}: {text}" for i, text in enumerate(summaries)])

    # Prompt con istruzioni più chiare e strutturate
    prompt = f"""
Sei un classificatore di testi. La tua funzione è assegnare UNA SOLA categoria a ciascun testo.

CATEGORIE VALIDE (usa ESATTAMENTE una di queste, con la stessa ortografia):
{categories_formatted}

TESTI DA CLASSIFICARE:
{texts_to_classify}

ISTRUZIONI IMPORTANTI:
1. Classifica ogni testo in ESATTAMENTE UNA delle categorie elencate sopra
2. Se non sei sicuro, scegli la categoria più appropriata tra quelle fornite
3. Non inventare nuove categorie
4. Rispondi con UNA RIGA PER TESTO nel formato "Testo N: categoria"
5. Devi classificare TUTTI i {len(summaries)} testi

FORMATO RISPOSTA RICHIESTO:
Testo 1: categoria
Testo 2: categoria
...
Testo {len(summaries)}: categoria

ATTENZIONE: La tua risposta deve contenere esattamente {len(summaries)} righe, una per ogni testo.
"""

    # Effettua la chiamata al modello
    response = llm_app.invoke(prompt)

    # Parsing della risposta
    lines = response.strip().split("\n")
    results = []

    # Mappa per tenere traccia di quali testi sono stati classificati
    classified_texts = {}

    # Analizza le risposte
    for line in lines:
        line = line.strip()
        # Cerca pattern "Testo N: categoria"
        if ":" in line:
            parts = line.split(":", 1)
            text_part = parts[0].strip()
            category = parts[1].strip()

            # Estrai il numero del testo
            text_index = None
            if text_part.lower().startswith("testo "):
                try:
                    text_index = int(text_part.lower().replace("testo ", "")) - 1
                except ValueError:
                    continue

            # Aggiungi solo se è un indice valido e la categoria è valida
            if text_index is not None and 0 <= text_index < len(summaries):
                if category in categories:
                    classified_texts[text_index] = category

    # Riempi l'array di risultati in ordine
    for i in range(len(summaries)):
        if i in classified_texts:
            results.append(classified_texts[i])
        else:
            # Se manca una classificazione, usa "other" come fallback
            results.append("other")

    # Verifica che ci siano esattamente tanti risultati quanti testi
    assert len(results) == len(summaries), f"Mismatch: {len(results)} results vs {len(summaries)} inputs"

    return results

In [ ]:
%%time

llm_tiny = Ollama(model="gemma2:2b", temperature=0)

batch_size = 70  # puoi aumentare finché non superi i limiti di token

results = []
for i in range(0, len(df_events_work), batch_size):
    batch = df_events_work.iloc[i:i+batch_size]
    batch_categories = classify_batch_local(llm_tiny, batch["summary"].tolist(), categories)
    results.extend(batch_categories)

# Verifica che lunghezze corrispondano
if len(results) != len(df_events_work):
    raise ValueError(f"Mismatch: {len(results)} results vs {len(df)} rows")

df_events_work["event_category_tiny"] = results

In [ ]:
df_pivot_cat = (
    df
    .loc[(df["calendar_name"] == "Pozz Work"), :]
    # .head()
    .merge(df_events_work, on="summary", how="left")
    #.head()
    .groupby([dict_view_available[view], "event_category_tiny"])
    # .size()
    .agg({"duration_minutes": "sum"})
    .sort_values("duration_minutes", ascending=False)
    .pivot_table(index="event_category_tiny", columns=dict_view_available[view], values="duration_minutes")
    .fillna(0)
)

In [ ]:
# Creazione della heatmap
fig = px.imshow(df_pivot_cat,
                labels=dict(x="Time", y="Summary", color="Valore"),
                x=df_pivot_cat.columns,
                y=df_pivot_cat.index,
                color_continuous_scale="Blues") # Puoi scegliere altre scale di colori

fig.update_xaxes(type='category')
fig.update_layout(title="Evoluzione attività svolte", title_x=0.5)

# Mostra la figura
fig.show()

In [ ]:
df_api_vs_local = (
    df_events_work
    #.head()
    .groupby(["event_category", "event_category_tiny"])
    .size()
    .to_frame("count")
    #.agg({"duration_minutes": "sum"})
    .sort_values("count", ascending=False)
    .pivot_table(index="event_category", columns="event_category_tiny", values="count")
    .fillna(0)
)

In [ ]:
# Creazione della heatmap
fig = px.imshow(df_api_vs_local,
                labels=dict(x="Summary API", y="Summary Ollama", color="Valore"),
                x=df_api_vs_local.columns,
                y=df_api_vs_local.index,
                color_continuous_scale="Blues") # Puoi scegliere altre scale di colori

fig.update_xaxes(type='category')
fig.update_layout(title="Matrice di Transizione", title_x=0.5)

# Mostra la figura
fig.show()

In [ ]:
# Creazione della heatmap con seaborn
plt.figure(figsize=(12, 10))  # Dimensione della figura

# Crea la heatmap
heatmap = sns.heatmap(df_api_vs_local,
                      annot=True,  # Mostra i valori in ogni cella
                      cmap="Blues",  # Usa la stessa scala di colori di Plotly
                      fmt=".2f",  # Formatta i numeri con 2 decimali
                      linewidths=.5,  # Linee tra le celle
                      cbar_kws={'label': 'Valore'})

# Imposta il titolo e regola la posizione
plt.title("Matrice di Transizione", fontsize=16)
plt.tight_layout()  # Migliora il layout

# Ruota le etichette se necessario per migliorare la leggibilità
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# Imposta le etichette degli assi
plt.xlabel("Summary Ollama")
plt.ylabel("Summary API")

# Mostra il grafico
plt.show()